# DDoS Auswertung

## Imports

In [2]:
import subprocess
from pathlib import Path
import polars as pl
import io
import pyarrow.parquet as pq

## Globale Parameter

In [3]:
SERVER_IP = "141.22.28.227"

In [4]:
SMALL_FIELDS = [
    "frame.number",
    "frame.time",
    "frame.time_epoch",
    "frame.len",
    "frame.protocols",
    "ip.proto",
    "ip.src",
    "ip.dst",
    "udp.srcport",
    "udp.dstport",
]
FIELDS = [
    "frame.number",
    "frame.time",
    "frame.time_epoch",
    "frame.len",
    "frame.protocols",
    "ip.proto",
    "ip.src",
    "ip.dst",
    "udp.srcport",
    "udp.dstport",
    "coap.block_length",
    "coap.block_payload",
    "coap.code",
    "coap.mid",
    "coap.opt.block_mflag",
    "coap.opt.block_number",
    "coap.opt.block_size",
    "coap.opt.ctype",
    "coap.opt.delta",
    "coap.opt.delta_ext",
    "coap.opt.desc",
    "coap.opt.end_marker",
    "coap.opt.etag",
    "coap.opt.length",
    "coap.opt.max_age",
    "coap.opt.name",
    "coap.opt.uri_path",
    "coap.opt.uri_path_recon",
    "coap.payload",
    "coap.payload_desc",
    "coap.payload_length",
    "coap.token_len",
    "coap.type",
    "coap.version",
]

## Parsen der Dateien

In [14]:
def pcap_to_parquet(input_file: Path, output_file: Path, batch_size: int = 500_000) -> None:
    schema_overrides = {
        "frame.time": pl.Datetime,
        "frame.time_epoch": pl.Float64,
        "frame.len": pl.Int64,
        "udp.srcport": pl.Int64,
        "udp.dstport": pl.Int64,
    }

    csv_path = output_file.with_suffix(".csv")
    cmd = ["tshark", "-r", str(input_file), "-T", "fields"]
    for f in SMALL_FIELDS:
        cmd += ["-e", f]
    cmd += ["-E", "header=y", "-E", "separator=,", "-E", "quote=d", "-E", "occurrence=f"]

    with open(csv_path, "wb") as f:
        result = subprocess.run(cmd, stdout=f, stderr=subprocess.PIPE)
    if result.returncode != 0:
        csv_path.unlink(missing_ok=True)
        raise RuntimeError(result.stderr.decode())

    lf = pl.scan_csv(csv_path, schema_overrides=schema_overrides, null_values=[""])

    writer = None
    try:
        for batch in lf.collect_batches(chunk_size=batch_size):
            table = batch.to_arrow()
            if writer is None:
                writer = pq.ParquetWriter(output_file, table.schema)
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()

    csv_path.unlink()

In [15]:
input_file = Path("data/raw/2025-08-13_15-38CEST_ddos-event_full-packets_100MiB.pcap.zst")
output_file = Path("data/interim/2025-08-13_15-38CEST_ddos-event_full-packets_100MiB.parquet")
df = pcap_to_parquet(input_file, output_file)

## Analyse der Daten

## Daten

In [7]:
lf = pl.scan_parquet(output_file)
df_head = lf.head(10).collect()
df_head

frame.number,frame.time,frame.time_epoch,frame.len,frame.protocols,ip.proto,ip.src,ip.dst,udp.srcport,udp.dstport
i64,datetime[μs],f64,i64,str,i64,str,str,i64,i64
1,2025-08-13 13:28:28.354835,1.7551e9,482,"""eth:ethertype:ip:udp:ntp""",17,"""68.168.214.83""","""141.22.28.227""",123,64219
2,2025-08-13 13:28:28.354835,1.7551e9,443,"""eth:ethertype:ip:udp:data""",17,"""85.113.53.151""","""141.22.28.227""",49928,15561
3,2025-08-13 13:28:28.354922,1.7551e9,1514,"""eth:ethertype:ip:data""",17,"""199.172.240.213""","""141.22.28.227""",null,null
4,2025-08-13 13:28:28.354922,1.7551e9,407,"""eth:ethertype:ip:udp:data""",17,"""102.152.183.249""","""141.22.28.227""",59480,23984
5,2025-08-13 13:28:28.355004,1.7551e9,619,"""eth:ethertype:ip:udp:xml""",17,"""66.170.190.104""","""141.22.28.227""",3702,32994
6,2025-08-13 13:28:28.355092,1.7551e9,940,"""eth:ethertype:ip:data""",17,"""160.250.5.142""","""141.22.28.227""",null,null
7,2025-08-13 13:28:28.355183,1.7551e9,541,"""eth:ethertype:ip:udp:ssdp""",17,"""114.230.110.29""","""141.22.28.227""",1900,52238
8,2025-08-13 13:28:28.355183,1.7551e9,407,"""eth:ethertype:ip:udp:data""",17,"""102.152.175.124""","""141.22.28.227""",39227,10445
9,2025-08-13 13:28:28.355183,1.7551e9,411,"""eth:ethertype:ip:udp:data""",17,"""41.226.182.16""","""141.22.28.227""",49188,53989


In [13]:
total_packets: int  = lf.select(pl.len()).collect().item()
total_bytes: int = lf.select(pl.col("frame.len").sum()).collect().item()

t_start = lf.select(pl.col("frame.time").min()).collect().item()
t_end = lf.select(pl.col("frame.time").max()).collect().item()
duration_s = (t_end - t_start).total_seconds()

print(f"Pakete gesamt : {total_packets:>12,}")
print(f"Bytes gesamt  : {total_bytes:>12,}  ({total_bytes/1e6:.1f} MB)")
print(f"Zeitraum      : {t_start}  →  {t_end}")
print(f"Dauer         : {duration_s:.1f} s")
print(f"Ø Paketrate   : {total_packets/duration_s:>10,.1f} pkt/s")
print(f"Ø Bitrate     : {total_bytes*8/duration_s/1e6:>10,.1f} Mbit/s")

Pakete gesamt :      186,212
Bytes gesamt  :   97,022,383  (97.0 MB)
Zeitraum      : 2025-08-13 13:28:28.354835  →  2025-08-13 13:28:41.138751
Dauer         : 12.8 s
Ø Paketrate   :   14,566.1 pkt/s
Ø Bitrate     :       60.7 Mbit/s


### Incoming vs. Outgoing

In [8]:
def split_direction(lf, server_ip):
    out_lf = lf.filter(pl.col("ip.src") == server_ip)
    in_lf = lf.filter(pl.col("ip.dst") == server_ip)
    foreign_lf = lf.filter(
        (pl.col("ip.dst") != server_ip) & (pl.col("ip.src") != server_ip)
    )
    null_lf = lf.filter(
        pl.col("ip.src").is_null() | pl.col("ip.dst").is_null()
    )
    return out_lf, in_lf, foreign_lf, null_lf

In [9]:
lf_out, lf_in, lf_foreign, lf_null = split_direction(lf, SERVER_IP)
len_out = lf_out.select(pl.len()).collect().item()
len_in = lf_in.select(pl.len()).collect().item()
len_foreign = lf_foreign.select(pl.len()).collect().item()
len_null = lf_null.select(pl.len()).collect().item()

print("Packets:", lf.select(pl.len()).collect().item()," = ", lf_in.select(pl.len()).collect().item(), "(incoming) + ", lf_out.select(pl.len()).collect().item(), "(outgoing)")
lf_null.collect().show(limit=None)

Packets: 186212  =  175764 (incoming) +  10408 (outgoing)


frame.number,frame.time,frame.time_epoch,frame.len,frame.protocols,ip.proto,ip.src,ip.dst,udp.srcport,udp.dstport
i64,datetime[μs],f64,i64,str,i64,str,str,i64,i64
15421,2025-08-13 13:28:28.931716,1.7551e9,60,"""eth:ethertype:arp""",null,null,null,null,null
24759,2025-08-13 13:28:29.338257,1.7551e9,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,546,547
30665,2025-08-13 13:28:29.647690,1.7551e9,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,546,547
34204,2025-08-13 13:28:29.847154,1.7551e9,68,"""eth:ethertype:vlan:llc:stp""",null,null,null,null,null
42596,2025-08-13 13:28:30.294619,1.7551e9,64,"""eth:llc:stp""",null,null,null,null,null
67561,2025-08-13 13:28:31.751741,1.7551e9,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,546,547
69277,2025-08-13 13:28:31.847368,1.7551e9,68,"""eth:ethertype:vlan:llc:stp""",null,null,null,null,null
70241,2025-08-13 13:28:31.901377,1.7551e9,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,546,547
74401,2025-08-13 13:28:32.166964,1.7551e9,571,"""eth:llc:cdp""",null,null,null,null,null


### Packets/Bytes per Second

In [10]:
df_in_per_sec = (
    lf_in.with_columns(pl.col("frame.time_epoch").cast(pl.Int64).alias("ts"))
    .group_by(pl.col("ts"))
    .agg([
        pl.len().alias("packets"),
        pl.col("frame.len").sum().alias("bytes"),
    ])
    .sort("ts")
    .collect()
)

df_in_per_sec.head(20)

ts,packets,bytes
i64,u64,i64
1755091708,16428,9238916
1755091709,19237,10892725
1755091710,17288,9487156
1755091711,15585,8178871
1755091712,14518,7512168
…,…,…
1755091717,11350,5824107
1755091718,11146,5681099
1755091719,9984,4938980


## 5. Top-Talker (meiste Pakete / meiste Bytes)

In [11]:
top_talkers = (
    lf.group_by("ip.src")
    .agg([
        pl.len().alias("packets"),
        pl.col("frame.len").sum().alias("bytes"),
    ])
    .sort("bytes", descending=True)
    .collect()
)

top_talkers.head(20)

ip.src,packets,bytes
str,u64,i64
"""141.22.28.227""",10408,4616742
"""49.176.212.229""",1685,812170
"""208.82.127.233""",1500,723000
"""173.245.226.45""",1400,674800
"""201.168.4.30""",1400,674800
…,…,…
"""167.88.48.3""",800,385600
"""38.88.6.90""",800,385600
"""201.168.193.30""",800,385600


In [12]:
df_pro_nb = (
    pl.read_csv("data/protocol-numbers-1.csv", ignore_errors=True)
      .select("Decimal", "Keyword")
).rename({"Keyword": "ip_proto_name", "Decimal": "ip_proto_num"})
df_pro_nb.head(10)

FileNotFoundError: No such file or directory (os error 2): data/protocol-numbers-1.csv

In [ ]:
df = (
    df_in
    .rename({c: c.replace(".", "_") for c in df_in.columns})  # Punkte → Unterstriche
    .with_columns([
        # Timestamp als datetime
        (pl.col("frame_time_epoch").cast(pl.Float64) * 1_000_000)
            .cast(pl.Int64)
            
            .cast(pl.Datetime("us"))
            .alias("timestamp"),

        # Einheitliche Ports (TCP oder UDP)
        pl.coalesce(["tcp_srcport", "udp_srcport"]).alias("src_port"),
        pl.coalesce(["tcp_dstport", "udp_dstport"]).alias("dst_port"),
    ])
    # translate number to name with iana protocol list
    .join(df_pro_nb, left_on="ip_proto", right_on="ip_proto_num", how="left")

    .drop(["frame_time_epoch", "ip_proto", "tcp_srcport", "tcp_dstport", "udp_srcport", "udp_dstport"])
)

print(df.shape)
df.show(10)

df_special = df.filter(
    pl.col("ip_proto_name") == "TCP"
    # ~pl.col("ip_proto_name").is_in(["tcp", "udp"])
)
print(df.shape)
df_special.show(10)


## 4. Protokollverteilung

In [ ]:
proto_dist = (
    df.with_columns(pl.col("l4_proto").fill_null("unknown"))
      .group_by("l4_proto")
      .agg([
          pl.len().alias("packets"),
          pl.col("frame_len").sum().alias("bytes"),
      ])
      .sort("packets", descending=True)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in zip(axes, ["packets", "bytes"], ["Pakete", "Bytes"]):
    ax.bar(proto_dist["l4_proto"], proto_dist[col])
    ax.set_title(f"Protokoll – {label}")
    ax.set_xlabel("Protokoll")
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

proto_dist

## 5. Zeitreihe – Paketrate

In [ ]:
BUCKET = "1s"  # Aggregations-Intervall: "100ms", "1s", "10s" ...

ts_rate = (
    df.sort("timestamp")
      .group_by_dynamic("timestamp", every=BUCKET)
      .agg([
          pl.len().alias("pkt_per_s"),
          (pl.col("frame_len").sum() * 8 / 1e6).alias("mbit_per_s"),
      ])
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax1.plot(ts_rate["timestamp"].to_list(), ts_rate["pkt_per_s"].to_list(), linewidth=0.8)
ax1.set_ylabel("Pakete/s")
ax1.set_title(f"Paketrate (Bucket: {BUCKET})")

ax2.plot(ts_rate["timestamp"].to_list(), ts_rate["mbit_per_s"].to_list(), color="orange", linewidth=0.8)
ax2.set_ylabel("Mbit/s")
ax2.set_xlabel("Zeit")

plt.tight_layout()
plt.show()

## 6. Top Quell-IPs (potenzielle Angreifer)

In [ ]:
TOP_N = 20

top_src = (
    df.filter(pl.col("ip_src").is_not_null())
      .group_by("ip_src")
      .agg([
          pl.len().alias("packets"),
          pl.col("frame_len").sum().alias("bytes"),
          pl.col("ip_dst").n_unique().alias("unique_dst"),
      ])
      .sort("packets", descending=True)
      .head(TOP_N)
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top_src["ip_src"].to_list()[::-1], top_src["packets"].to_list()[::-1])
ax.set_xlabel("Pakete")
ax.set_title(f"Top {TOP_N} Quell-IPs")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

top_src

## 7. Top Ziel-Ports

In [ ]:
top_dstport = (
    df.filter(pl.col("dst_port").is_not_null())
      .group_by(["dst_port", "l4_proto"])
      .agg(pl.len().alias("packets"))
      .sort("packets", descending=True)
      .head(15)
      .with_columns(
          (pl.col("dst_port").cast(pl.Utf8) + "/" + pl.col("l4_proto")).alias("port_proto")
      )
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(top_dstport["port_proto"].to_list(), top_dstport["packets"].to_list())
ax.set_xlabel("Port/Protokoll")
ax.set_ylabel("Pakete")
ax.set_title("Top Ziel-Ports")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

top_dstport.drop("port_proto")

## 8. Paketgrößenverteilung

In [ ]:
sizes = df["frame_len"].drop_nulls().to_list()

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(sizes, bins=100, edgecolor="none")
ax.set_xlabel("Paketgröße (Bytes)")
ax.set_ylabel("Häufigkeit")
ax.set_title("Paketgrößenverteilung")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

df.select(pl.col("frame_len").describe())

## 9. Export

In [ ]:
out = Path("output")
out.mkdir(exist_ok=True)

df.write_parquet(out / "packets.parquet")
top_src.write_csv(out / "top_src_ips.csv")

print("Exportiert nach:", out.resolve())

## 7. Amplification-Metriken (BAF/PAF), optional

Falls Request/Response anhand `coap.code` oder Port-Richtung unterschieden werden können, hier die Grundstruktur. An eure CoAP-Logik anpassen (z. B. Server-Port als Filter, GET-Request vs. Response).

In [ ]:
SERVER_PORT = 5683  # ggf. anpassen

requests = lf.filter(pl.col("udp.dstport") == SERVER_PORT)
responses = lf.filter(pl.col("udp.srcport") == SERVER_PORT)

req_stats = requests.select([
    pl.len().alias("req_packets"),
    pl.col("frame.len").sum().alias("req_bytes"),
]).collect()

resp_stats = responses.select([
    pl.len().alias("resp_packets"),
    pl.col("frame.len").sum().alias("resp_bytes"),
]).collect()

baf = resp_stats["resp_bytes"][0] / req_stats["req_bytes"][0]
paf = resp_stats["resp_packets"][0] / req_stats["req_packets"][0]

print(f"BAF (Bandwidth Amplification Factor): {baf:.2f}")
print(f"PAF (Packet Amplification Factor):   {paf:.2f}")

BAF (Bandwidth Amplification Factor): 47270.96
PAF (Packet Amplification Factor):   4011.00
